In [1]:
import torch;

import torch.nn as nn

import torch.nn.functional as F

with open("george_washington.txt", "r", encoding="utf-8") as f:
         text = f.read()
    
    # extract all unique chars
chars = sorted(set(text))

# calculate vocabulary size
vocab_size = len(chars)

# create character-to-integer mapping
string_to_int = { ch: i for i, ch in enumerate(chars)}
# create integer to character mapping
int_to_string = { i: ch for i, ch in enumerate(chars)}

def encode(s):
     return [string_to_int[c] for c in s]
    
def decode(indices):
    return "".join(int_to_string[i] for i in indices)



class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    
    
    
    def forward(self, index, target=None):
        logits = self.token_embedding_table(index)

        if target is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.reshape(B*T, C)
            target = target.reshape(B*T)
            loss = F.cross_entropy(logits, target)

        return logits, loss


    def generate(self, index, max_new_tokens):
            # index is (B, T) array if indices in the current context
            for _ in range(max_new_tokens):
                #get the predictions
                logits, loss = self.forward(index)
                # focuss only the last time step
                logits = logits[:, -1, :] # becomes (B, C)
                # apply softmax to get the probability
                probs = F.softmax(logits, dim=-1) # B, C
                # sample from distribution
                index_next = torch.multinomial(probs, num_samples=1) # (B, 1)
                # append sampled to the runned sequence
                index = torch.cat((index, index_next), dim=1)
            
            return index

# Detect GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
# Create model
model = BigramLanguageModel(vocab_size=vocab_size)
m = model.to(device)
# initial context
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())

print(generated_chars)

KeyError: 96